# Character‑Level LSTM Text Generation (Improved)

## Overview
This notebook trains a character‑level LSTM model on Nietzsche’s works and generates new text with adjustable temperature.  
Key considerations:
- Integer encoding + embedding layer (reduces memory).
- Data generator for streaming batches.
- Model with dropout, recurrent dropout, and gradient clipping (improves stability).
- Single `fit` call with checkpoints and early stopping.
- Fixed temperature loop (seed is reset each time).
- Better output formatting.


## 1. Imports and Setup

In [11]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
import random
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# For reproducibility
seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)
random.seed(seed)

## 2. Load and Preprocess Data
We download the Nietzsche text, convert it to lowercase (optional), and create character mappings

In [12]:
# Download the text
path = tf.keras.utils.get_file('nietzsche.txt', origin='https://s3.amazonaws.com/text-datasets/nietzsche.txt')
text = open(path, 'r', encoding='utf-8').read().lower()
print('Corpus length:', len(text))

# Unique characters
chars = sorted(set(text))
char2idx = {ch: i for i, ch in enumerate(chars)}
idx2char = {i: ch for i, ch in enumerate(chars)}
vocab_size = len(chars)
print('Vocabulary size:', vocab_size)

# Hyperparameters
maxlen = 60          # sequence length
step = 3             # stride
batch_size = 128

Corpus length: 600893
Vocabulary size: 57


### Data Generator
Instead of storing the entire one‑hot encoded array in memory, we create a generator that yields batches of integer sequences and their targets. This is memory‑efficient and allows processing of arbitrarily large texts.

In [13]:
def generate_sequences(text, maxlen, step, batch_size):
    """Yield batches of (input_seq, target_char) as integer indices."""
    sequences = []
    next_chars = []
    for i in range(0, len(text) - maxlen, step):
        sequences.append(text[i:i + maxlen])
        next_chars.append(text[i + maxlen])
        if len(sequences) == batch_size:
            # Convert to integer arrays
            x = np.zeros((batch_size, maxlen), dtype=np.int32)
            y = np.zeros((batch_size, vocab_size), dtype=np.int32)
            for j, seq in enumerate(sequences):
                for t, ch in enumerate(seq):
                    x[j, t] = char2idx[ch]
                y[j, char2idx[next_chars[j]]] = 1
            yield x, y
            sequences = []
            next_chars = []
    # Last batch (smaller)
    if sequences:
        x = np.zeros((len(sequences), maxlen), dtype=np.int32)
        y = np.zeros((len(sequences), vocab_size), dtype=np.int32)
        for j, seq in enumerate(sequences):
            for t, ch in enumerate(seq):
                x[j, t] = char2idx[ch]
            y[j, char2idx[next_chars[j]]] = 1
        yield x, y

# Create a generator for training
train_gen = generate_sequences(text, maxlen, step, batch_size)

## 3. Build the Model
We use an Embedding layer (reduces dimensionality) followed by an LSTM with dropout, and a Dense softmax output.

In [14]:
embedding_dim = 256
lstm_units = 512

model = models.Sequential([
    layers.Embedding(vocab_size, embedding_dim, input_length=maxlen),
    layers.LSTM(lstm_units, dropout=0.2, recurrent_dropout=0.2),
    layers.Dense(vocab_size, activation='softmax')
])

# Compile with Adam and gradient clipping
optimizer = optimizers.Adam(learning_rate=0.001, clipnorm=1.0)
model.compile(loss='categorical_crossentropy', optimizer=optimizer)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## 4. Callbacks

We set up:

- ModelCheckpoint to save the best weights.

- EarlyStopping to stop when validation loss stops improving (here we use the training loss as a proxy).

- Custom callback to generate text at the end of each epoch for monitoring.

In [15]:
# Save the best model based on loss
checkpoint = callbacks.ModelCheckpoint(
    filepath='best_model.keras',
    monitor='loss',
    save_best_only=True,
    verbose=1
)

# Stop if loss doesn't improve for 5 epochs
early_stop = callbacks.EarlyStopping(monitor='loss', patience=5, verbose=1)

# Custom callback for text generation
class TextGenerator(callbacks.Callback):
    def __init__(self, seed_text, char2idx, idx2char, maxlen, temperatures=[0.2,0.5,1.0,1.2], generate_len=400):
        self.seed_text = seed_text
        self.char2idx = char2idx
        self.idx2char = idx2char
        self.maxlen = maxlen
        self.temperatures = temperatures
        self.generate_len = generate_len

    def on_epoch_end(self, epoch, logs=None):
        print(f'\n--- Epoch {epoch+1} ---')
        for temp in self.temperatures:
            print(f'Temperature: {temp}')
            generated = self.seed_text
            sys.stdout.write(generated)
            for _ in range(self.generate_len):
                x = np.zeros((1, self.maxlen), dtype=np.int32)
                for t, ch in enumerate(generated):
                    x[0, t] = self.char2idx.get(ch, 0)
                preds = model.predict(x, verbose=0)[0]
                # --- robust sampling ---
                preds = np.asarray(preds).astype('float64')
                preds = np.clip(preds, 1e-10, 1.0)
                preds = np.log(preds) / temp
                exp_preds = np.exp(preds)
                probs = exp_preds / np.sum(exp_preds)
                next_idx = np.random.choice(len(probs), p=probs)
                # -----------------------
                next_char = self.idx2char[next_idx]
                generated = generated[1:] + next_char
                sys.stdout.write(next_char)
            print('\n' + '-'*50)

    def sample(self, preds, temperature):
        preds = np.asarray(preds).astype('float64')
        # Add small epsilon to avoid log(0)
        preds = np.clip(preds, 1e-10, 1.0)
        preds = np.log(preds) / temperature
        exp_preds = np.exp(preds)
        preds = exp_preds / np.sum(exp_preds)
        probas = np.random.multinomial(1, preds, 1)
        return np.argmax(probas)

# Select a random seed for generation
start_index = random.randint(0, len(text) - maxlen - 1)
seed = text[start_index:start_index + maxlen]
print('Seed text:', seed)

text_gen_callback = TextGenerator(seed, char2idx, idx2char, maxlen)

Seed text:  of all that women write about "woman," we may well have
con


## 5. Training
We now train the model for up to 60 epochs, using the generator. The callbacks will handle saving, early stopping, and periodic generation.

In [16]:
# Calculate number of steps per epoch
total_sequences = (len(text) - maxlen) // step + 1
steps_per_epoch = total_sequences // batch_size
print(f'Total sequences: {total_sequences}, Steps per epoch: {steps_per_epoch}')

history = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    epochs=60,
    callbacks=[checkpoint, early_stop, text_gen_callback],
    verbose=1
)

Total sequences: 200278, Steps per epoch: 1564
Epoch 1/60
1564/1564 ━━━━━━━━━━━━━━━━━━━━ 0s 471ms/step - loss: 2.3991
Epoch 1: loss improved from None to 2.09380, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras

--- Epoch 1 ---
Temperature: 0.2
 of all that women write about "woman," we may well have
consument the same in the say in the san the consention the sance of the saction of the say in the same the say and the sand the saction the sensing the same in the same feeling in the sand the say and the saint and the say the self the san the same the san the matter the say in the same the cansent in the sand the san the say the saint the same the say and the sanstion and the same the say in the sa
--------------------------------------------------
Temperature: 0.5
 of all that women write about "woman," we may well have
consumpes and the man the partuin that for at is with the pare and the frater which the sample, and by the say the so all everyther 

## 6. Final Generation
After training (or loading the best model), we can generate text with any temperature.

In [17]:
# Load the best model
best_model = models.load_model('best_model.keras')

def generate_text(seed, length=400, temperature=1.0):
    """Generate text from a seed using the trained model."""
    generated = seed
    sys.stdout.write(generated)
    for _ in range(length):
        # Encode the current sequence
        x = np.zeros((1, maxlen), dtype=np.int32)
        for t, ch in enumerate(generated):
            x[0, t] = char2idx.get(ch, 0)
        # Predict probabilities for next character
        preds = best_model.predict(x, verbose=0)[0]
        # Sample with temperature
        preds = np.asarray(preds).astype('float64')
        preds = np.clip(preds, 1e-10, 1.0)
        preds = np.log(preds) / temperature
        exp_preds = np.exp(preds)
        probs = exp_preds / np.sum(exp_preds)
        # Use choice to avoid multinomial precision issues
        next_idx = np.random.choice(len(probs), p=probs)
        next_char = idx2char[next_idx]
        generated = generated[1:] + next_char
        sys.stdout.write(next_char)
    print()

# Example: generate with different temperatures
print('--- Generation with temperature 0.5 ---')
generate_text(seed, temperature=0.5)

print('\n--- Generation with temperature 1.2 ---')
generate_text(seed, temperature=1.2)

--- Generation with temperature 0.5 ---
 of all that women write about "woman," we may well have
constatis wat they nave beenson on pheroodd---that it have baidd and chaidd at loadn of sunder at they saining and chemand compled at they naterin pompain and chave beend of so that it have buen---whate been---nat they nom not they not they nave baind--are seemaning in pherenon on so that it have been on more not that it have been on that it have beennon on so gheren---that it have buen---at sairtain

--- Generation with temperature 1.2 ---
 of all that women write about "woman," we may well have
consequosiling. for
sont that for
so not there nound--maind--that comtation have been--who not vay main phomont in plowt that its loer and
contleninn evereryod firse
whateress.,) whate inspiantain payd wherere wisthalled seemang and coudd as leengndodiby port of state inseequed aw !e, tho gaind-that us adongaäly pertaind eostams feein of sunsuind allod arequed of sourd--that they knoventaring in pu

## 7. Discussion

This notebook demonstrates:

Efficient data handling with integer encoding and a generator.

Stable training via dropout, gradient clipping, and Adam.

Automatic checkpointing to avoid losing progress.

Cleaner generation with reset seed and better formatting.

Customizable temperature for controlled randomness.

This starting notebook provides a solid foundation to further experiment with hyperparameters (sequence length, embedding size, number of layers) and sampling strategies (e.g., top‑k sampling) to refine the generated text.